# Automated Code Review with LangGraph
### Notebook 3 of 3 — Advanced Patterns

Unit testing, LLM-as-Judge quality gating, and extensions to explore.

| Notebook | Focus |
|----------|-------|
| 01 — Foundations | State, nodes, graph build |
| 02 — Demo & Patterns | All 4 paths, streaming, persistence, token cost |
| **03 — Advanced Patterns** (this one) | Unit testing, LLM-as-Judge, retry loops |

**Prerequisites:** Run the setup cell below first.

In [1]:
# ── Setup: import all nodes and rebuild both graphs ─────────────
import os, json
from pathlib import Path
from typing import TypedDict, Annotated, Optional, List
import operator

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv(Path.cwd() / ".env")
NOTEBOOK_DIR = Path.cwd()

# Production modules
from state import (PRReviewState, SimpleReviewOutput, SummarizeFindingsOutput,
                   FixItem, QualityJudgeOutput, merge_usage)
from nodes import (read_pr_file, route_review_type, routing_edge,
                   simple_review_node, quality_analysis_node, security_review_node,
                   aggregate_node, output_guardrail, blocked_review_node,
                   summarize_findings_node, final_decision_node, return_final_answer,
                   full_review_start, quality_judge_node, quality_judge_edge,
                   judge_llm)

# Shared LLM (used in unit tests and inline demos)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# ── v1 graph (used in unit test helpers) ────────────────────
def _build_v1():
    b = StateGraph(PRReviewState)
    b.add_node("read_pr",             read_pr_file)
    b.add_node("route_review",        route_review_type)
    b.add_node("simple_review",       simple_review_node)
    b.add_node("full_review_start",   full_review_start)
    b.add_node("security_review",     security_review_node)
    b.add_node("quality_analysis",    quality_analysis_node)
    b.add_node("aggregate",           aggregate_node)
    b.add_node("blocked_review",      blocked_review_node)
    b.add_node("summarize_findings",  summarize_findings_node)
    b.add_node("final_decision",      final_decision_node)
    b.add_node("return_final_answer", return_final_answer)
    b.add_edge(START, "read_pr")
    b.add_edge("read_pr", "route_review")
    b.add_conditional_edges("route_review", routing_edge,
        {"simple": "simple_review", "full": "full_review_start", "error": "return_final_answer"})
    b.add_edge("full_review_start", "security_review")
    b.add_edge("full_review_start", "quality_analysis")
    b.add_edge("security_review",  "aggregate")
    b.add_edge("quality_analysis", "aggregate")
    b.add_edge("aggregate", "summarize_findings")
    b.add_conditional_edges("summarize_findings", output_guardrail,
        {"proceed": "final_decision", "block": "blocked_review"})
    b.add_edge("blocked_review", "final_decision")
    b.add_edge("simple_review",  "final_decision")
    b.add_edge("final_decision",      "return_final_answer")
    b.add_edge("return_final_answer", END)
    return b.compile()

code_review_app = _build_v1()
print("✅ code_review_app (v1) ready")

✅ code_review_app (v1) ready


## Step 22 — Unit Testing Nodes in Isolation

### The key insight: nodes are pure functions

Every node has the signature `fn(state: dict) -> dict`. This means you can **call any node directly** with a handcrafted dict — no graph compilation, no LLM calls for deterministic nodes, no waiting for parallel branches to finish.

```python
# Instead of running the whole graph:
result = code_review_app.invoke(initial_state)

# Call the node directly with just the fields it needs:
result = read_pr_file({"pr_file_path": "files/code_changes_simple.txt", ...})
assert "pr_content" in result
```

### What to test without LLMs

These nodes and functions are **fully deterministic** — unit-testable with zero API calls:

| Function | Test for |
|----------|---------|
| `read_pr_file` | Valid path → `pr_content` populated; missing path → `errors` set |
| `routing_edge` | Returns exact string matching `review_type` in state |
| `output_guardrail` | `"BLOCKING"` → `"block"`; `"NON-BLOCKING"` → `"proceed"`; neither → `"proceed"` |
| `report_guardrail` | Always returns `"complete"` |
| `merge_usage` | `{a:1} + {a:2}` → `{a:3}`; handles missing keys |
| `FixItem` | Rejects missing fields; accepts valid issue/solution/explanation |

### What to test with LLMs (integration tests)

Nodes that call the LLM (`route_review_type`, `simple_review_node`, etc.) need real API calls. Test these by asserting on the **shape** of the output (keys present, types correct) rather than the content, which varies between runs.

In [2]:
_base = {
    "pr_file_path": "", "pr_content": "", "review_type": "",
    "quality_findings": None, "security_findings": None,
    "summary": None, "simple_review": None, "final_decision": None,
    "quality_retry_count": 0, "judge_score": None, "judge_reason": None,
    "errors": [], "tokens_used": {}, "messages": []
}

print("=" * 60)
print("UNIT TESTS — nodes called directly, no graph compilation")
print("=" * 60)

# --- Test 1: read_pr_file — valid file ---
result = read_pr_file({**_base, "pr_file_path": "files/code_changes_simple.txt"})
assert "pr_content" in result and result["pr_content"], "pr_content should be populated"
assert not result.get("errors"), "no errors expected for valid file"
print("✅  1. read_pr_file (valid path)   → pr_content populated, no errors")

# --- Test 2: read_pr_file — missing file ---
result = read_pr_file({**_base, "pr_file_path": "files/does_not_exist.txt"})
assert result.get("errors"), "errors should be set for missing file"
assert "pr_content" not in result, "pr_content must not be set on error"
print("✅  2. read_pr_file (missing path) → errors populated, no pr_content")

# --- Test 3: routing_edge reads review_type ---
for rt in ("simple", "full", "error"):
    assert routing_edge({**_base, "review_type": rt}) == rt
print("✅  3. routing_edge               → returns correct label for all 3 review types")

# --- Test 4: output_guardrail — BLOCKING ---
result = output_guardrail({**_base, "security_findings": "HIGH risk SQL injection. BLOCKING"})
assert result == "block"
print("✅  4. output_guardrail (BLOCKING) → returns 'block'")

# --- Test 5: output_guardrail — NON-BLOCKING ---
result = output_guardrail({**_base, "security_findings": "LOW risk minor issues. NON-BLOCKING"})
assert result == "proceed"
print("✅  5. output_guardrail (NON-BLOCKING) → returns 'proceed'")

# --- Test 6: output_guardrail — no findings ---
result = output_guardrail({**_base, "security_findings": "No issues found."})
assert result == "proceed"
print("✅  6. output_guardrail (clean)    → returns 'proceed'")

# --- Test 7: merge_usage reducer accumulates correctly ---
a = {"input_tokens": 100, "output_tokens": 50,  "total_tokens": 150}
b = {"input_tokens": 200, "output_tokens": 100, "total_tokens": 300}
merged = merge_usage(a, b)
assert merged == {"input_tokens": 300, "output_tokens": 150, "total_tokens": 450}
print("✅  7. merge_usage reducer         → sums token counts correctly")

# --- Test 8: FixItem validates required fields ---
try:
    FixItem(issue="SQL injection", solution="Use parameterised queries", explanation="Prevents data breach")
    print("✅  8. FixItem model               → accepts valid issue/solution/explanation")
except Exception as e:
    print(f"❌  8. FixItem model               → unexpected error: {e}")

print(f"\n✅ All 8 unit tests passed — {len(_base)} state fields, 0 LLM calls")
print("   (quality_judge_edge tests run in Step 23 — after that function is defined)")

UNIT TESTS — nodes called directly, no graph compilation
✅  1. read_pr_file (valid path)   → pr_content populated, no errors
✅  2. read_pr_file (missing path) → errors populated, no pr_content
✅  3. routing_edge               → returns correct label for all 3 review types
✅  4. output_guardrail (BLOCKING) → returns 'block'
✅  5. output_guardrail (NON-BLOCKING) → returns 'proceed'
✅  6. output_guardrail (clean)    → returns 'proceed'
✅  7. merge_usage reducer         → sums token counts correctly
✅  8. FixItem model               → accepts valid issue/solution/explanation

✅ All 8 unit tests passed — 14 state fields, 0 LLM calls
   (quality_judge_edge tests run in Step 23 — after that function is defined)


## Step 23 — LLM-as-Judge: Quality Review Gating

### The LLM-as-Judge pattern

An LLM-as-Judge is a dedicated model call whose sole purpose is to **evaluate the output of another LLM** and decide whether it meets a quality bar. The judge's verdict drives routing — not a keyword match or a heuristic.

```
quality_analysis ──► quality_judge ──[score >= 7]──► aggregate (proceed)
        ↑                   │
        └──[score < 7, retries < 2]
```

### Why a separate judge model?

This step uses `gpt-4o-mini` as the judge — deliberately different from `gpt-4o` used for the analysis nodes:

| Node | Model | Why |
|------|-------|-----|
| `quality_analysis` | `gpt-4o` | Deep code reasoning required |
| `quality_judge` | `gpt-4o-mini` | Scoring 1–10 is a lightweight task |
| `security_review` | `gpt-4o` | Security reasoning requires capability |

**Right-sizing models per task** reduces cost. The judge reads one analysis and returns a score — it does not need full `gpt-4o` capability.

### Self-evaluation vs cross-model judging

Using a different model as judge avoids **self-evaluation bias** — the risk that a model rates its own output highly regardless of quality. `gpt-4o-mini` judging `gpt-4o` output is a practical form of cross-model validation.

### Retry with a stricter prompt

When the judge scores below 7, `quality_analysis_node_v2` re-runs with a stricter system prompt — explicitly instructing the LLM to be more thorough. `quality_retry_count` tracks retries and caps at 2 to prevent infinite loops.

### New state fields

```python
quality_retry_count: int           # tracks retries — caps at 2
judge_score:         Optional[int] # 1-10 score from judge
judge_reason:        Optional[str] # one-sentence explanation
```

### Extended graph segment

```
full_review_start ──► quality_analysis ──► quality_judge ──[score >= 7]──► aggregate
                              ↑                   │
                              └──[score < 7, retries <= 2]──┘  (max 2 retries)
```

> **Key distinction:** `output_guardrail` (Step 7) uses a keyword check — fast, deterministic, zero tokens. `quality_judge` uses an LLM call — flexible, semantic, costs tokens. Use guardrails for binary pass/fail on structured output; use LLM judges for nuanced quality scoring.

In [3]:
from pydantic import BaseModel, Field as PydanticField

# ------------------------------------------------------------------
# Judge LLM — gpt-4o-mini (lightweight scoring task)
# ------------------------------------------------------------------
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# ------------------------------------------------------------------
# Pydantic output model for the judge
# ------------------------------------------------------------------
class QualityJudgeOutput(BaseModel):
    score:  int = PydanticField(description="Thoroughness score 1-10")
    reason: str = PydanticField(description="One sentence explaining the score")


# ------------------------------------------------------------------
# quality_analysis_node_v2 — retry-aware, stricter prompt on re-run
# ------------------------------------------------------------------
def quality_analysis_node_v2(state: PRReviewState) -> dict:
    retry = state.get("quality_retry_count", 0)
    if retry > 0:
        system_prompt = """You are a Senior Developer performing a rigorous code quality review.
A previous review of this PR was rated insufficient. Be more thorough this time.

Analyse the PR diff and provide a detailed report covering:

1. STYLE ISSUES        — naming, formatting, PEP8 violations
2. POTENTIAL BUGS      — logic errors, edge cases, exception handling gaps
3. SEVERITY            — label each issue CRITICAL, MAJOR, or MINOR

Reference specific line numbers. Do not omit any issues."""
        print(f"🔄 Quality Analysis retry {retry}/2 — stricter prompt active")
    else:
        system_prompt = """You are a Senior Developer performing a code quality review.

Analyse the PR diff and provide a structured report covering:

1. STYLE ISSUES        — naming conventions, formatting, code organisation
2. POTENTIAL BUGS      — logic errors, edge cases, exception handling gaps
3. SEVERITY            — label each issue CRITICAL, MAJOR, or MINOR

Be specific. Reference line numbers or code snippets where relevant."""

    try:
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"PR Diff to review:\n\n{state['pr_content']}"),
        ])
        content = response.content
        usage   = response.usage_metadata or {}
    except Exception as e:
        content = f"[ERROR] LLM call failed: {e}"
        usage   = {}

    label = f" (retry {retry})" if retry > 0 else ""
    print(f"👨‍💻 Quality Analysis complete{label}")

    return {
        "quality_findings": content,
        "tokens_used": {
            "input_tokens":  usage.get("input_tokens",  0),
            "output_tokens": usage.get("output_tokens", 0),
            "total_tokens":  usage.get("total_tokens",  0),
        },
        "messages": [f"[quality_analysis] complete ({len(content)} chars){label}"],
    }


# ------------------------------------------------------------------
# quality_judge_node — LLM-as-Judge using gpt-4o-mini
# ------------------------------------------------------------------
def quality_judge_node(state: PRReviewState) -> dict:
    structured_llm = judge_llm.with_structured_output(QualityJudgeOutput, include_raw=True)
    system_prompt = """You are a code review quality assessor.

Rate the thoroughness of the following code quality analysis on a scale of 1-10:
- 8-10: Comprehensive — covers style, bugs, severity levels, specific line references
- 5-7:  Adequate — covers main issues but lacks specificity or misses some areas
- 1-4:  Insufficient — too brief, vague, or misses obvious issues

Return a score (int 1-10) and a one-sentence reason."""

    try:
        raw_result = structured_llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"Quality analysis to evaluate:\n\n{state.get('quality_findings', '')}"),
        ])
        judge_out = raw_result["parsed"]
        usage     = raw_result["raw"].usage_metadata or {}
        score     = judge_out.score
        reason    = judge_out.reason
    except Exception as e:
        score  = 8
        reason = f"[ERROR] Judge failed: {e} — defaulting to proceed"
        usage  = {}

    retry_count     = state.get("quality_retry_count") or 0
    new_retry_count = retry_count + 1 if score < 7 else retry_count

    if score >= 7:
        verdict = f"score={score}/10 — passed"
    elif new_retry_count <= 2:
        verdict = f"score={score}/10 — retry {new_retry_count}/2"
    else:
        verdict = f"score={score}/10 — retry cap reached, proceeding"

    print(f"⚖️  Quality Judge: {verdict}")

    return {
        "judge_score":         score,
        "judge_reason":        reason,
        "quality_retry_count": new_retry_count,
        "tokens_used": {
            "input_tokens":  usage.get("input_tokens",  0),
            "output_tokens": usage.get("output_tokens", 0),
            "total_tokens":  usage.get("total_tokens",  0),
        },
        "messages": [f"[quality_judge] {verdict} — {reason[:80]}"],
    }


def quality_judge_edge(state: PRReviewState) -> str:
    """Returns 'retry' to re-run quality_analysis, or 'proceed' to continue."""
    score       = state.get("judge_score") or 10
    retry_count = state.get("quality_retry_count") or 0
    if score < 7 and retry_count <= 2:
        return "retry"
    return "proceed"


# ------------------------------------------------------------------
# Build code_review_app_v2 — extends the original graph with the judge
# ------------------------------------------------------------------
v2_builder = StateGraph(PRReviewState)

# Same nodes as code_review_app...
v2_builder.add_node("read_pr",             read_pr_file)
v2_builder.add_node("route_review",        route_review_type)
v2_builder.add_node("simple_review",       simple_review_node)
v2_builder.add_node("full_review_start",   full_review_start)
v2_builder.add_node("security_review",     security_review_node)
v2_builder.add_node("aggregate",           aggregate_node)
v2_builder.add_node("blocked_review",      blocked_review_node)
v2_builder.add_node("summarize_findings",  summarize_findings_node)
v2_builder.add_node("final_decision",      final_decision_node)
v2_builder.add_node("return_final_answer", return_final_answer)

# ...plus the judge nodes
v2_builder.add_node("quality_analysis",    quality_analysis_node_v2)
v2_builder.add_node("quality_judge",       quality_judge_node)

# Edges — same as v1 except quality_analysis → quality_judge → [retry|aggregate]
v2_builder.add_edge(START, "read_pr")
v2_builder.add_edge("read_pr", "route_review")
v2_builder.add_conditional_edges(
    "route_review", routing_edge,
    {"simple": "simple_review", "full": "full_review_start", "error": "return_final_answer"}
)
v2_builder.add_edge("full_review_start", "security_review")
v2_builder.add_edge("full_review_start", "quality_analysis")

# Judge gate: quality_analysis → quality_judge → retry or proceed
v2_builder.add_edge("quality_analysis", "quality_judge")
v2_builder.add_conditional_edges(
    "quality_judge", quality_judge_edge,
    {"retry": "quality_analysis", "proceed": "aggregate"}
)

v2_builder.add_edge("security_review", "aggregate")

# Always synthesise first — gives recommendations on ALL paths including REJECT
v2_builder.add_edge("aggregate", "summarize_findings")
v2_builder.add_conditional_edges(
    "summarize_findings", output_guardrail,
    {"proceed": "final_decision", "block": "blocked_review"}
)
v2_builder.add_edge("blocked_review", "final_decision")
v2_builder.add_edge("simple_review",       "final_decision")
v2_builder.add_edge("final_decision",      "return_final_answer")
v2_builder.add_edge("return_final_answer", END)

code_review_app_v2 = v2_builder.compile()
print("✅ code_review_app_v2 compiled")
print("\nGraph change vs v1:")
print("  quality_analysis → quality_judge → [retry → quality_analysis | proceed → aggregate]")
print(f"\nJudge model: gpt-4o-mini   (analysis model: gpt-4o)")

✅ code_review_app_v2 compiled

Graph change vs v1:
  quality_analysis → quality_judge → [retry → quality_analysis | proceed → aggregate]

Judge model: gpt-4o-mini   (analysis model: gpt-4o)


In [4]:
print("🚀 Running code_review_app_v2 on: code_changes_needswork.txt")
print("Expected: full → quality_analysis → quality_judge → aggregate → NON-BLOCKING\n")
print("=" * 60)

v2_state: PRReviewState = {
    "pr_file_path":        "files/code_changes_needswork.txt",
    "pr_content":          "",
    "review_type":         "",
    "quality_findings":    None,
    "security_findings":   None,
    "summary":             None,
    "simple_review":       None,
    "final_decision":      None,
    "quality_retry_count": 0,
    "judge_score":         None,
    "judge_reason":        None,
    "errors":              [],
    "tokens_used":         {},
    "messages":            [],
}

result_v2 = code_review_app_v2.invoke(v2_state)

print("\n" + "=" * 60)
print(f"⚖️  Judge score:   {result_v2.get('judge_score')}/10")
print(f"📝 Judge reason:  {result_v2.get('judge_reason')}")
print(f"🔄 Retries fired: {result_v2.get('quality_retry_count', 0)}")

print("\n" + "=" * 60)
print("MESSAGE LOG:")
for i, msg in enumerate(result_v2["messages"], 1):
    print(f"  {i:2}. {msg[:120]}")

print("\n" + "=" * 60)
print("TOKEN USAGE:")
for k, v in result_v2.get("tokens_used", {}).items():
    print(f"  {k}: {v}")

# ------------------------------------------------------------------
# Unit tests for quality_judge_edge (defined in cell above)
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("UNIT TESTS — quality_judge_edge")
print("=" * 60)

_b = {**{k: None for k in result_v2}, "errors": [], "tokens_used": {}, "messages": []}

result = quality_judge_edge({**_b, "judge_score": 8,  "quality_retry_count": 0})
assert result == "proceed", f"expected 'proceed', got '{result}'"
print("✅  9. quality_judge_edge (pass)      → score=8 returns 'proceed'")

result = quality_judge_edge({**_b, "judge_score": 7,  "quality_retry_count": 0})
assert result == "proceed"
print("✅ 10. quality_judge_edge (threshold) → score=7 returns 'proceed'")

result = quality_judge_edge({**_b, "judge_score": 4,  "quality_retry_count": 1})
assert result == "retry"
print("✅ 11. quality_judge_edge (retry)     → score=4, retries=1 returns 'retry'")

result = quality_judge_edge({**_b, "judge_score": 4,  "quality_retry_count": 3})
assert result == "proceed"
print("✅ 12. quality_judge_edge (cap)       → score=4, retries=3 returns 'proceed'")

result = quality_judge_edge({**_b, "judge_score": None, "quality_retry_count": 0})
assert result == "proceed"
print("✅ 13. quality_judge_edge (no score)  → None score defaults to 'proceed'")

print("\n✅ All 5 judge edge tests passed — 0 LLM calls")

🚀 Running code_review_app_v2 on: code_changes_needswork.txt
Expected: full → quality_analysis → quality_judge → aggregate → NON-BLOCKING



👨‍💻 Quality Analysis complete


⚖️  Quality Judge: score=8/10 — passed



FINAL DECISION (Full Code Review Tool): REQUEST CHANGES




FINAL ANSWER
[Full Code Review Tool] REQUEST CHANGES

While the security review indicates no vulnerabilities, the quality analysis was incomplete due to a system error. It is essential to re-run the quality analysis to ensure there are no undetected issues that could affect the functionality or maintainability of the code. Once this is addressed, the code can be considered for merging.

────────────────────────────────────────────────────────────
REQUIRED FIXES (1 item — must address before merge):
────────────────────────────────────────────────────────────

  1. Issue:      Quality analysis was not completed due to a technical error.
     Solution:   Re-run the quality analysis to ensure there are no overlooked issues in code quality or functionality.
     Why:        Without a complete quality analysis, there is a risk of merging code that may have functional or stylistic issues that could affect maintainability or performance.

─────────────────────────────────────────────────────


FINAL DECISION (Full Code Review Tool): REQUEST CHANGES


FINAL ANSWER
[Full Code Review Tool] REQUEST CHANGES

While the security review indicates no vulnerabilities, the quality analysis was not completed due to a technical error. It is essential to re-run the quality analysis to ensure there are no overlooked issues that could affect the code's functionality or maintainability. Addressing this will ensure the code meets the necessary quality standards before merging.

────────────────────────────────────────────────────────────
REQUIRED FIXES (1 item — must address before merge):
────────────────────────────────────────────────────────────

  1. Issue:      Quality analysis was not completed due to a technical error.
     Solution:   Re-run the quality analysis to ensure there are no overlooked issues in code quality or functionality.
     Why:        Without a complete quality analysis, there is a risk of merging code that may have functional or stylistic issues that could affect 

## Key Takeaways & Further Learning

---

### Patterns Reference

| Pattern | Node / Location | Core idea |
|---------|----------------|-----------|
| **LLM-as-router** | `route_review_type` | LLM classifies input; edge function returns node name |
| **Error propagation in state** | `read_pr_file` | Return `{"errors": [...]}` instead of raising -- router detects and routes |
| **Fan-out parallelism** | `full_review_start` | Multiple `add_edge(X, ...)` from the same source runs nodes concurrently |
| **Fan-in synchronisation** | `aggregate_node` | All parallel edges converge on one join node; LangGraph waits for all |
| **Guardrail edge function** | `output_guardrail` | Conditional edge that inspects state without an LLM call or state update |
| **Hard-stop node** | `blocked_review_node` | Writes a REJECT verdict so all downstream paths converge uniformly |
| **Structured output + tokens** | `simple_review_node`, `summarize_findings_node` | `with_structured_output(..., include_raw=True)` gives Pydantic + `usage_metadata` in one call |
| **Custom reducer** | `merge_usage` | `Annotated[dict, merge_usage]` sums token counts correctly across parallel nodes |
| **Append-only log** | `messages` field | `Annotated[list, operator.add]` builds a lightweight, zero-cost audit trail |
| **Terminal node** | `return_final_answer` | Separates final computation from presentation; all paths converge here |
| **State persistence** | Step 13 | `json.dump(result, ...)` for offline inspection and auditing |
| **Graph checkpointing** | Step 20 | `SqliteSaver` checkpointer enables run history, replay, and resumable flows |
| **LLM-as-Judge** | `quality_judge_node` | Separate model (`gpt-4o-mini`) scores another LLM's output; drives retry routing |
| **Right-sizing models** | Step 23 | Match model capability to task weight — lightweight scoring doesn't need the strongest model |
| **Retry loop with cap** | `quality_judge_edge` | Conditional edge loops back with a stricter prompt; `quality_retry_count` prevents infinite loops |

---

### Production Modules

The graph logic is extracted into two standalone Python files you can import directly:

```python
from state import PRReviewState, SimpleReviewOutput, SummarizeFindingsOutput, merge_usage
from nodes import read_pr_file, route_review_type, quality_analysis_node, security_review_node
```

| File | Contents |
|------|----------|
| `state.py` | `PRReviewState` TypedDict, Pydantic output models, `merge_usage` reducer |
| `nodes.py` | All 15 node and edge functions — no notebook dependencies |

---

### Further Reading

- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/) -- official docs, concepts, and API reference
- [LangGraph How-To Guides](https://langchain-ai.github.io/langgraph/how-tos/) -- parallelism, subgraphs, streaming, persistence
- [LangChain Structured Output](https://python.langchain.com/docs/concepts/structured_outputs/) -- `with_structured_output` patterns
- [Pydantic v2 Docs](https://docs.pydantic.dev/) -- Field definitions, validators, model serialisation

---

### Extensions to Explore

1. **Add a `needs_human_review` path**: Extend `output_guardrail` to return a third option, `"human"`, when security findings exist but are all LOW risk. Route to a new node that formats a triage summary for a human reviewer.

2. **Replace file input with GitHub API**: Modify `read_pr_file` to fetch the diff from the GitHub Pull Requests API instead of a local file. The rest of the graph stays unchanged -- this is the modularity benefit of isolated nodes.

3. **Cost budgeting**: Add a `token_budget: int` field to `PRReviewState`. Before each LLM call, check if `tokens_used["total_tokens"] + estimated_tokens > token_budget`. If over budget, skip the call and route to `return_final_answer` with a "budget exceeded" message.

4. **Extend the simple path**: Currently the simple path skips security review entirely. Add a fast security scan (a single-prompt check for obvious patterns like hardcoded secrets) between `simple_review` and `final_decision`.

5. **Add a security judge**: Apply the same LLM-as-Judge pattern to `security_review_node` — have `gpt-4o-mini` score the security analysis before it reaches `output_guardrail`.